In [1]:
import numpy as np
import gymnasium as gym
from gymnasium.utils.env_checker import check_env
import stable_baselines3 as sb3


In [ ]:
class customEnv(gym.Env):

    def __init__(self):
        super().__init__()

        self.size = 10
        self.pos = np.array([1, 1], dtype=np.float32)
        self.size = np.array([1, 1], dtype=np.float32)
        self.material = 0

        self.target_pos = np.array([4, 3], dtype=np.float32)
        self.target_size = np.array([4, 4], dtype=np.float32)
        self.target_material = 1

        self.observation_space = gym.spaces.Dict({
            'pos': gym.spaces.Box(low=-self.size, high=self.size, shape=(2, ), dtype=np.float32),
            'size': gym.spaces.Box(low=-self.size, high=self.size, shape=(2, ), dtype=np.float32),
            'material': gym.spaces.Discrete(2, dtype=int)
        })

        self.action_space = gym.spaces.Dict({
            'pos': gym.spaces.Box(low=-self.size, high=self.size, shape=(2,), dtype=np.float32),
            'size': gym.spaces.Box(low=-self.size, high=self.size, shape=(2,), dtype=np.float32),
            'material': gym.spaces.Discrete(2)
        })

    def _get_obs(self):
        return {
            'pos': self.pos,
            'size': self.size,
            'material': self.material
        }

    def _get_info(self):
        return {
            'distance': (np.linalg.norm(self.pos - self.target_pos, ord=1) + np.linalg.norm(self.size - self.target_size, ord=1) + abs(self.material - self.target_material))
        }

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)

        self.pos = np.array([1, 1], dtype=np.float32)
        self.size = np.array([1, 1], dtype=np.float32)
        self.material = 0

        self.target_pos = np.array([4, 3], dtype=np.float32)
        self.target_size = np.array([4, 4], dtype=np.float32)
        self.target_material = 1

        obs = self._get_obs()
        info = self._get_info()

        return obs, info

    def step(self, action):
        #Action will be a vector with numbers

        terminated = True if np.array_equal(self.pos, self.target_pos) and np.array_equal(self.size, self.target_size) and self.material == self.target_material else False

        truncated = False
        reward = -self._get_info()['distance']

        obs = self._get_obs()
        info = self._get_info()

        return obs, reward, terminated, truncated, info

In [96]:
gym.register(id='customEnv-v0', entry_point=customEnv)

env = gym.make('customEnv-v0')

check_env(env.unwrapped)

/home/stelfano/anaconda3/envs/RL/lib/python3.13/site-packages/gymnasium/envs/registration.py:636: UserWarning: WARN: Overriding environment customEnv-v0 already in registry.
  logger.warn(f"Overriding environment {new_spec.id} already in registry.")


# Model definition

This is a simple model in torchRL as done in the first tutorial

In [97]:
from collections import defaultdict

import matplotlib.pyplot as plt
import torch
from tensordict.nn import TensorDictModule
from tensordict.nn.distributions import NormalParamExtractor
from torch import nn

import torchrl
from torchrl.collectors import SyncDataCollector
from torchrl.data.replay_buffers import ReplayBuffer
from torchrl.data.replay_buffers.samplers import SamplerWithoutReplacement
from torchrl.data.replay_buffers.storages import LazyTensorStorage
from torchrl.envs import (
    Compose,
    DoubleToFloat,
    ObservationNorm,
    StepCounter,
    TransformedEnv,
    CatTensors
)
from torchrl.envs.libs.gym import GymEnv
from torchrl.envs.utils import check_env_specs, ExplorationType, set_exploration_type
from torchrl.modules import ProbabilisticActor, TanhNormal, ValueOperator
from torchrl.objectives import ClipPPOLoss
from torchrl.objectives.value import GAE
from tqdm import tqdm


In [98]:
device = torch.device("cpu")
num_cells = 256  # number of cells in each layer i.e. output dim.
lr = 3e-4
max_grad_norm = 1.0

In [99]:
frames_per_batch = 1000
total_frames = 100000

In [100]:
sub_batch_size = 64
num_epochs = 10
clip_epsilon = (
    0.2  
)
gamma = 0.99
lmbda = 0.95
entropy_eps = 1e-4

In [101]:
torchEnv = torchrl.envs.GymWrapper(env)

torchEnv = TransformedEnv(torchEnv, Compose(CatTensors(in_keys=['pos', 'size', 'material'], out_key='obs')))

In [102]:
print("observation_spec:", torchEnv.observation_spec)
print("reward_spec:", torchEnv.reward_spec)
print("input_spec:", torchEnv.input_spec)
print("action_spec (as defined by input_spec):", torchEnv.action_spec)

observation_spec: Composite(
    obs: UnboundedDiscrete(
        shape=torch.Size([6]),
        space=ContinuousBox(
            low=Tensor(shape=torch.Size([6]), device=cpu, dtype=torch.int64, contiguous=True),
            high=Tensor(shape=torch.Size([6]), device=cpu, dtype=torch.int64, contiguous=True)),
        device=cpu,
        dtype=torch.int64,
        domain=discrete),
    device=None,
    shape=torch.Size([]),
    data_cls=None)
reward_spec: UnboundedContinuous(
    shape=torch.Size([1]),
    space=ContinuousBox(
        low=Tensor(shape=torch.Size([1]), device=cpu, dtype=torch.float32, contiguous=True),
        high=Tensor(shape=torch.Size([1]), device=cpu, dtype=torch.float32, contiguous=True)),
    device=cpu,
    dtype=torch.float32,
    domain=continuous)
input_spec: Composite(
    full_state_spec: Composite(device=None, shape=torch.Size([]), data_cls=None),
    full_action_spec: Composite(
        material: OneHot(
            shape=torch.Size([2]),
            space=C

In [103]:
check_env_specs(torchEnv)

AssertionError: The dtypes of the real and fake tensordict don't match for key obs. Got fake=torch.int64 and real=torch.float32.

In [104]:
rollout = torchEnv.rollout(3)
print("rollout:", rollout)

rollout: TensorDict(
    fields={
        done: Tensor(shape=torch.Size([3, 1]), device=cpu, dtype=torch.bool, is_shared=False),
        material: Tensor(shape=torch.Size([3, 2]), device=cpu, dtype=torch.int64, is_shared=False),
        next: TensorDict(
            fields={
                done: Tensor(shape=torch.Size([3, 1]), device=cpu, dtype=torch.bool, is_shared=False),
                obs: Tensor(shape=torch.Size([3, 6]), device=cpu, dtype=torch.float32, is_shared=False),
                reward: Tensor(shape=torch.Size([3, 1]), device=cpu, dtype=torch.float32, is_shared=False),
                terminated: Tensor(shape=torch.Size([3, 1]), device=cpu, dtype=torch.bool, is_shared=False),
                truncated: Tensor(shape=torch.Size([3, 1]), device=cpu, dtype=torch.bool, is_shared=False)},
            batch_size=torch.Size([3]),
            device=None,
            is_shared=False),
        obs: Tensor(shape=torch.Size([3, 6]), device=cpu, dtype=torch.float32, is_shared=False)

In [105]:
def print_rollout(rollout):
    for key, value in rollout.items():
        print(f"{key}: {value}")

In [106]:
print_rollout(rollout)

done: tensor([[False],
        [False],
        [False]])
terminated: tensor([[False],
        [False],
        [False]])
truncated: tensor([[False],
        [False],
        [False]])
obs: tensor([[1., 0., 1., 1., 1., 1.],
        [1., 0., 1., 1., 1., 1.],
        [1., 0., 1., 1., 1., 1.]])
material: tensor([[0, 1],
        [0, 1],
        [0, 1]])
pos: tensor([[-0.7276,  0.7653],
        [-0.7249, -0.6110],
        [-0.3577, -0.9537]])
size: tensor([[-0.9446, -0.4976],
        [-0.2751,  0.1538],
        [ 0.2076,  0.1000]])
next: TensorDict(
    fields={
        done: Tensor(shape=torch.Size([3, 1]), device=cpu, dtype=torch.bool, is_shared=False),
        obs: Tensor(shape=torch.Size([3, 6]), device=cpu, dtype=torch.float32, is_shared=False),
        reward: Tensor(shape=torch.Size([3, 1]), device=cpu, dtype=torch.float32, is_shared=False),
        terminated: Tensor(shape=torch.Size([3, 1]), device=cpu, dtype=torch.bool, is_shared=False),
        truncated: Tensor(shape=torch.Size(

In [135]:
pos = torchEnv.action_spec['pos']
size = torchEnv.action_spec['size']
material = torchEnv.action_spec['material']

obs_spec = torchEnv.observation_spec
print(obs_spec)
obs_spec_size = sum(spec.shape[-1] for spec in obs_spec.values())
print("observation spec size:", obs_spec_size)

print(pos)
print(size)
print(material)

act_spec_size = pos.shape[0] + size.shape[0] + material.shape[0]
print("total action spec size:", act_spec_size)

material.shape[0] + pos.shape[0]
print(material.shape[0])

Composite(
    obs: UnboundedDiscrete(
        shape=torch.Size([6]),
        space=ContinuousBox(
            low=Tensor(shape=torch.Size([6]), device=cpu, dtype=torch.int64, contiguous=True),
            high=Tensor(shape=torch.Size([6]), device=cpu, dtype=torch.int64, contiguous=True)),
        device=cpu,
        dtype=torch.int64,
        domain=discrete),
    device=None,
    shape=torch.Size([]),
    data_cls=None)
observation spec size: 6
BoundedContinuous(
    shape=torch.Size([2]),
    space=ContinuousBox(
        low=Tensor(shape=torch.Size([2]), device=cpu, dtype=torch.float32, contiguous=True),
        high=Tensor(shape=torch.Size([2]), device=cpu, dtype=torch.float32, contiguous=True)),
    device=cpu,
    dtype=torch.float32,
    domain=continuous)
BoundedContinuous(
    shape=torch.Size([2]),
    space=ContinuousBox(
        low=Tensor(shape=torch.Size([2]), device=cpu, dtype=torch.float32, contiguous=True),
        high=Tensor(shape=torch.Size([2]), device=cpu, dtype=t

In [136]:
num_cells = 256
import torch.nn as nn

class actor_net(nn.Module):
    def __init__(self):
        super().__init__()
        self.l1 = nn.Linear(obs_spec_size, num_cells, device=device)
        self.tanh = nn.Tanh()
        self.l2 = nn.Linear(num_cells, num_cells, device=device)
        self.l3 = nn.Linear(num_cells, num_cells, device=device)
        self.l4 = nn.Linear(num_cells, 2 * act_spec_size, device=device)
        self.out = NormalParamExtractor()

    def forward(self, obs):
        x = self.l1(obs)
        x = self.tanh(x)
        x = self.l2(x)
        x = self.tanh(x)
        x = self.l3(x)
        x = self.tanh(x)
        x = self.l4(x)
        print(x.shape)
        
        return self.out(x)

In [137]:

actor = actor_net()
sampleobservation = torchEnv.observation_spec.rand()
print("sample observation:", sampleobservation['material'], sampleobservation['pos'], sampleobservation['size'])

#merge the observation into a single vector
def merge_observation(obs):
    return torch.cat((obs['material'], obs['pos'], obs['size']))

#use the actor to get the action distribution parameters
merged_obs = merge_observation(sampleobservation)
print("merged observation:", merge_observation(sampleobservation))
action_params = actor(merged_obs)
print("action parameters:", action_params)


KeyError: 'key "material" not found in TensorDict with keys [\'obs\']'

In [118]:
p_module = TensorDictModule(
    actor, in_keys=["obs"], out_keys=["loc", "scale"]
)

In [130]:
rollout = torchEnv.rollout(1)
print(rollout['obs'])

res = actor(rollout['obs'])
print(res)

tensor([[1., 0., 1., 1., 1., 1.]])
torch.Size([1, 12])
(tensor([[-0.1458, -0.1416, -0.1323, -0.0454,  0.0980, -0.0025]],
       grad_fn=<SplitBackward0>), tensor([[0.9948, 1.0923, 0.9543, 0.9742, 1.0238, 0.9914]],
       grad_fn=<ClampMinBackward0>))


In [121]:
# Extract low and high bounds for each continuous action component
low = np.concatenate([
    torchEnv.action_spec['pos'].space.low,
    torchEnv.action_spec['size'].space.low,
    np.zeros(torchEnv.action_spec['material'].shape, dtype=np.float32)
])
high = np.concatenate([
    torchEnv.action_spec['pos'].space.high,
    torchEnv.action_spec['size'].space.high,
    np.ones(torchEnv.action_spec['material'].shape, dtype=np.float32) * (torchEnv.action_spec['material'].space.n - 1)
])

policy_module = ProbabilisticActor(
    module=p_module,
    spec=torchEnv.action_spec,
    in_keys=["loc", "scale"],
    out_keys=list(torchEnv.action_spec.keys()),  # Ensure out_keys match the action spec keys
    distribution_class=TanhNormal,
    distribution_kwargs={
        "low": low,
        "high": high,
    },
    return_log_prob=True,
    # we'll need the log-prob for the numerator of the importance weights
)

In [150]:
print(torchEnv.action_spec.keys())
policy_module(rollout['obs'])

_CompositeSpecKeysView(keys=['material', 'pos', 'size'])
torch.Size([1, 12])


ValueError: zip() argument 2 is shorter than argument 1